In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_6 import BioJepa, BioJepaConfig
from dataloader_v0_6 import PretrainLoader, AlignmentLoader, TrainingLoader
from training_v0_6 import create_model, load_feature_banks, run_pretraining, run_alignment, run_full_training, train_linear_decoder, maybe_compile
from config_v0_6 import PretrainConfig, AlignmentConfig, FullTrainingConfig, DecoderConfig, DataConfig
from evals.evals import EvalContext, run_pretraining_evals, run_alignment_evals, run_full_model_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('/home/ubuntu/data/v0_6')
ref_root = Path('/home/ubuntu/data/reference_data')

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoints',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
# Model architecture
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=40.5,
    std_coeff=40.5,
    cov_coeff=1.62,
    pert_latent_dim= 320,
    pert_mode_dim= 64,
)

# Training configs
pt_cfg = PretrainConfig(epochs=1, lr=1e-3, batch_size=128) 
align_cfg = AlignmentConfig(epochs=100, lr=4e-3, batch_size=32)
full_cfg = FullTrainingConfig(epochs=1, predictor_lr=1e-3, batch_size=32) 
decoder_cfg = DecoderConfig(epochs=1, lr=1e-3, batch_size=16) 

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 7,979,266
ACpredictor: 10,036,224
PerturbationComposer: 1,478,976


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_full_final.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [6]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': pt_cfg.batch_size, 'seed': SEED
})
pt_eval_results = run_pretraining_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'pretraining_eval_report.json')
pt_eval_results

Using cuda
found 121 shards for split test


batch_invariance: Extracting embeddings: 100%|███████████████████████| 2420/2420 [06:36<00:00,  6.10it/s]


Training classifiers...
batch_invariance: Batch=0.0907 (34.1x), Pert=0.0773 (83.7x)
batch_invariance summary: global_ratio=0.851, within_dataset_macro_ratio=0.784
Loading KEGG_2021_Human...
  320 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
gene_embedding_pathways: KEGG sil=-0.0716, Reactome sil=-0.0830
essential_gene_prediction: Pearson=0.2216, AUROC=0.6302
found 121 shards for split test


cell_type_probing: Extracting embeddings: 100%|██████████████████████| 2420/2420 [06:25<00:00,  6.28it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.9927 (3.0x chance), Macro F1=0.9827
found 121 shards for split test


...

embedding_consistency: Extracting embeddings: 100%|██████████████████| 2420/2420 [06:26<00:00,  6.27it/s]


embedding_consistency: Computing intra-distances for 1084 perturbations...
embedding_consistency: Computing 5000 inter-distances...
embedding_consistency: Intra=17.0817, Inter=14.4929, Ratio=0.85x
embedding_consistency: Computing for dataset adamson...
embedding_consistency: Computing for dataset k562e_raw...
embedding_consistency: Computing for dataset k562gw...
embedding_consistency: Computing for dataset norman...
embedding_consistency: Computing for dataset sciplex...
found 121 shards for split test


latent_space_health: Extracting embeddings: 100%|████████████████████| 2420/2420 [06:22<00:00,  6.32it/s]


latent_space_health: Eff_dim_90=45/256, Mean_var=0.5095, Isotropy=0.000083
Saved report to /home/ubuntu/data/v0_6/eval_results/pretraining_eval_report.json


{'batch_invariance': {'config': {'samples': 309760,
   'embedding_dim': 256,
   'num_batches': 376,
   'num_perturbations': 1084},
  'batch_classifier': {'accuracy': 0.09074767561983471,
   'chance': 0.0026595744680851063,
   'above_chance_ratio': 34.121126033057855},
  'perturbation_classifier': {'accuracy': 0.07725335743801653,
   'chance': 0.0009225092250922509,
   'above_chance_ratio': 83.74263946280992},
  'invariance_ratio': 0.8512984702952686,
  'by_dataset': {'k562e_raw': {'config': {'samples': 49066,
     'embedding_dim': 256,
     'num_batches': 48,
     'num_perturbations': 286},
    'batch_classifier': {'accuracy': 0.0920114122681883,
     'chance': 0.020833333333333332,
     'above_chance_ratio': 4.416547788873039},
    'perturbation_classifier': {'accuracy': 0.015691868758915834,
     'chance': 0.0034965034965034965,
     'above_chance_ratio': 4.487874465049929},
    'invariance_ratio': 0.17054263565891473},
   'k562gw': {'config': {'samples': 178474,
     'embedding_dim'

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [12]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': align_cfg.batch_size, 'seed': SEED
})
align_eval_results = run_alignment_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'alignment_eval_report.json')
align_eval_results

Using cuda
Loaded 10797 v0.6 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 320])
Encoded chemical sequences: torch.Size([188, 320])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 320])
seq_to_target_retrieval: dna_mrr=0.0013
cross_modality_target_consistency: Within=0.2576, Between=0.0841, Ratio=3.06x
seq_target_gap_analysis: dna_gap=0.87
paired_alignment_quality: dna_sim=0.1206
mode_sensitivity: Classification_acc=0.5524 (3.9x chance)
fusion_quality: Fused_var=0.4480, Seq_var=0.3900, Target_var=0.0376
missing_data_robustness: Fused_MRR=0.0053, Seq_only=0.0019, Target_only=1.0000
found 121 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|██████████████████| 500/500 [00:03<00:00, 164.42it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0676, target_only=0.2364, fused=0.1784
Saved report to /home/ubuntu/data/v0_6/eval_results/alignment_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.0013275378271070334,
    'median_rank': 4497.0,
    'mean_rank': 4626.14702285768,
    'n_queries': 10631,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.0002821935847991722,
     '5': 0.0004703226413319537,
     '10': 0.0014109679239958611,
     '20': 0.0026338067914589407,
     '50': 0.008183613959175995}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 915,
   'n_within_pairs': 1452,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.2576046884059906,
   'between_target_sim': 0.08410928859282285,
   'consistency_ratio': 3.062737691826968}},
 'seq_target_gap_analysis': {'target_variance': 11.264479637145996,
  'n_targets': 9975,
  'dna': {'seq_variance': 135.24725341796875,
   'centroid_distance': 6.140118598937988,
   'mean_within_seq': 14.427490794058313,
   'mean_seq_to_target': 12.586881637573242,
   'gap_ratio': 0.8724234738556832,
   'n_sequen

In [13]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [14]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
decoder.load_state_dict(decoder_sd)

<All keys matched successfully>

In [15]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': full_cfg.batch_size, 'seed': SEED, 'inference_shard_size': 20000,
})
full_eval_results = run_full_model_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'full_model_eval_report.json')
full_eval_results

Using cuda
found 121 shards for split test


Running test inference:   0%|                                                   | 0/9680 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:  16%|██████▍                                 | 1555/9680 [06:22<19:32,  6.93it/s]/home/ubuntu/code/biojepa_unified/biojepa/evals/evals.py:537: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference: 100%|████████████████████████████████████████| 9680/9680 [41:38<00:00,  3.87it/s]


Aggregated 1085 perturbations, 309760 samples, 16 shards
  adamson: 9 perturbations, 5331 samples
  k562e_raw: 286 perturbations, 49066 samples
  k562gw: 1053 perturbations, 178474 samples
  norman: 8 perturbations, 2686 samples
  sciplex: 14 perturbations, 74203 samples
Cached test inference to /home/ubuntu/data/v0_6/test_inference_cache (16 shards)
expression_prediction: Pearson=0.9839, R2=0.9644, Centroid_acc=0.0341
gene_level_analysis: Dir_acc=0.9889, Top50_acc=0.7763


perturbation_retrieval (dna): 100%|██████████████████████████████████| 100/100 [1:17:23<00:00, 46.44s/it]


perturbation_retrieval (dna): MRR=0.0005


perturbation_retrieval (chemical): 100%|█████████████████████████████████| 13/13 [00:09<00:00,  1.34it/s]


perturbation_retrieval (chemical): MRR=0.0364
uncertainty_calibration: ECE=0.5718, Monotonicity=22.22%
Encoded DNA sequences: torch.Size([11643, 320])
Encoded chemical sequences: torch.Size([188, 320])
Encoded protein targets: torch.Size([9975, 320])
action_vector_pathways DNA: sil=-0.33890336751937866
Loaded 10797 v0.6 alignment pairs
moa_matching: Within=0.8277, Between=0.7251, Ratio=1.142x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
combination_perturbation: 8 combo perts, 2686 samples, 8 additive baseline, 6 GI-labeled
dose_response: monotonicity=47.62%, spearman=-0.1512
Saved report to /home/ubuntu/data/v0_6/eval_results/full_model_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1085,
   'genes': 10000,
   'test_samples': 309760},
  'sample_level': {'mse': 0.20359253280460335,
   'pearson_r_top20': 0.8694456461339609},
  'perturbation_level': {'r2_all_genes': {'mean': 0.9644321550422,
    'median': 0.9707128405570984},
   'r2_top50_degs': {'mean': 0.8159585150705505, 'median': 0.8487882614135742},
   'mse': {'mean': 0.00690451031550765, 'median': 0.0064394716173410416},
   'pearson_all_genes': {'mean': 0.9838728804742136,
    'median': 0.9869765043258667},
   'pearson_delta_all_genes': {'mean': 0.21642846134126462,
    'median': 0.21578755974769592},
   'pearson_top50_degs': {'mean': 0.41962209965793357,
    'median': 0.43616023659706116}},
  'centroid_accuracy': 0.034101382488479264,
  'vs_baseline': {'beat_rate': 0.009216589861751152, 'n_evaluated': 1085},
  'severity': {'pearson_r': 0.7381852865219116,
   'spearman_r': 0.6379918221108529},
  'error_by_magnitude': {'0-0.25': {'mae': 0.06139799952507

In [16]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()